### **Loading PDF** , remove <mark>, chunking

In [4]:
import re
import pymupdf4llm
from langchain_text_splitters import MarkdownHeaderTextSplitter

pdf_path = "source/USA_Employee_Handbook-Freely_Available.pdf"

md_text = pymupdf4llm.to_markdown(pdf_path)

clean_md = re.sub(r"</?mark>", "", md_text)  # Removes <mark> and </mark>
clean_md = re.sub(r"<[^>]+>", "", clean_md)   # Strip any remaining residual HTML tags

# 3. Configure 3-level header splitter
headers_to_split_on = [
    ("#", "parent_header"),
    ("##", "child_header"),
    ("###", "sub_child_header"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on, strip_headers=False
)

raw_chunks = markdown_splitter.split_text(clean_md)


chunks = []
for chunk in raw_chunks:
    if len(chunk.page_content.strip()) <= 50:
        continue

    meta = chunk.metadata
    meta.setdefault("parent_header", "")
    meta.setdefault("child_header", "")
    meta.setdefault("sub_child_header", "")

    chunk.metadata = meta
    chunks.append(chunk)


In [7]:
chunks[5]

Document(metadata={'parent_header': 'Employee Handbook', 'child_header': 'Employment basics', 'sub_child_header': 'Equal opportunity employment'}, page_content='### Equal opportunity employment  \n[ _Company’s name_ ] is an equal opportunity employer. We don’t tolerate discrimination against protected characteristics (gender, age, sexual orientation, race, nationality, ethnicity, religion, disability, veteran status.) We want all employees (including executives and HR) to treat others with respect and professionalism. In practice, this means that we:  \n- [ _Hire and promote people based on skills, experience or potential and try to_ _reduce bias in every process (e.g. through structured interviews.)_ ]  \n- [ _Make accommodations to help people with disabilities move about safely on our_ _premises and use our products, services and equipment_ .]  \n- [ _Use inclusive, diversity-sensitive language in all official documents, signs and job_ _ads_ .]  \n- [ _Conduct diversity and communic

### **Creating Embeddings**

In [6]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv()

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=768,  # As my documet size is less than 30 pages and each document is clearly defined under sub header I have uses 768 dimentions (defailt 1536) 
)

In [97]:
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000002DBC379CC50>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000002DBC378DA60>, model='text-embedding-3-small', dimensions=768, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

### **Creating Pinecode**

In [98]:
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone()

index_name = "employee-rag-pipeline"

if index_name not in [index.name for index in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=768,  # MUST match the 768 dimensions above
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

vector_store = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=index_name,
    namespace="chunks",
)

In [99]:
sample = vector_store.similarity_search("sick leave", k=1)

print(sample[0])

page_content='### Sick leave  
We offer [ _one week_ ] of [ _paid_ ] sick leave. In states or countries where employees are entitled to a greater number of sick leave days by law, we will follow that law. You can take sick leave to recover from short-term illness, injuries, mental issues and other indisposition. If you have the flu or other contagious disease, please use your sick days.  
If you become sick, inform your manager and send a sick leave request [ _through our HRIS._ ] You may take a partial day off or work from home, but we advise you to rest and recuperate for a day before returning to work.  
Use your PTO or arrange for a flexible work schedule if you want to attend routine health care (e.g. doctor’s/dentist’s appointments.)  
Occasionally, we may ask you to submit a physician’s note or other medical certification and/or complete a sick leave form. We will do this for insurance purposes if you are absent for more than [ _three days_ ] of sick leave.  
#### Long-term illn

### **Creating embedding for headders**

In [100]:
from langchain_core.documents import Document

header_docs = []
seen_headers = set()

for chunk in chunks:
    parent = chunk.metadata.get("parent_header", "").strip()
    child = chunk.metadata.get("child_header", "").strip()
    sub_child = chunk.metadata.get("sub_child_header", "").strip()

    if not child and not sub_child:
        continue

    combo_key = (parent, child, sub_child)
    if combo_key in seen_headers:
        continue
    seen_headers.add(combo_key)

    
    if sub_child:
        searchable_text = (
            f"Policy Topic: {sub_child}. Category: {child}. "
            f"Document: {parent}. Details, rules, and guidelines regarding {sub_child}."
        )
    else:
        searchable_text = (
            f"Policy Category: {child}. Document: {parent}. "
            f"General rules and guidelines under {child}."
        )

    header_docs.append(
        Document(
            page_content=searchable_text,
            metadata={
                "parent_header": parent,
                "child_header": child,
                "sub_child_header": sub_child,
            },
        )
    )


In [103]:
header_docs[4]

Document(metadata={'parent_header': 'Employee Handbook', 'child_header': 'Employment basics', 'sub_child_header': 'Equal opportunity employment'}, page_content='Policy Topic: Equal opportunity employment. Category: Employment basics. Document: Employee Handbook. Details, rules, and guidelines regarding Equal opportunity employment.')

In [104]:

from langchain_pinecone import PineconeVectorStore


headers_vectorstore = PineconeVectorStore.from_documents(
    documents=header_docs,
    embedding=embeddings,
    index_name=index_name,
    namespace="headers",
)

In [106]:
test_query = "can an employee wear smart casual"

results = headers_vectorstore.similarity_search_with_score(test_query, k=1)


In [107]:
results

[(Document(id='56e0319f-350f-4f03-b240-040cf3d10267', metadata={'child_header': 'Employee Code of Conduct', 'parent_header': 'Employee Handbook', 'sub_child_header': 'Dress code'}, page_content='Policy Topic: Dress code. Category: Employee Code of Conduct. Document: Employee Handbook. Details, rules, and guidelines regarding Dress code.'),
  0.475557238)]

### **Chunking and embedding humanresourcespolicy odc**

In [5]:
import re
import pymupdf4llm
from langchain_text_splitters import MarkdownHeaderTextSplitter

pdf_path = "source/humanresourcespolicy.pdf"
doc_id = "nestle_hr_policy"

md_text = pymupdf4llm.to_markdown(pdf_path)

clean_md = re.sub(r"</?mark>", "", md_text)
clean_md = re.sub(r"<[^>]+>", "", clean_md)

normalized_lines = []
for line in clean_md.split("\n"):
    stripped = line.strip()
    if stripped.startswith("# ") and not stripped.startswith("## "):
        line = "#" + line  # turns '# Joining Nestlé' into '## Joining Nestlé'
    normalized_lines.append(line)

normalized_md = "\n".join(normalized_lines)

headers_to_split_on = [
    ("##", "child_header"),
    ("###", "sub_child_header"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on, strip_headers=False
)

raw_chunks = markdown_splitter.split_text(normalized_md)

chunks2 = []
for chunk in raw_chunks:
    if len(chunk.page_content.strip()) <= 30:
        continue

    meta = chunk.metadata
    meta["parent_header"] = "Human Resources Policy"
    meta.setdefault("child_header", "")
    meta.setdefault("sub_child_header", "")
    meta["doc_source"] = doc_id

    chunk.metadata = meta
    chunks2.append(chunk)


In [ ]:
chunks2[4]

Document(metadata={'child_header': 'Joining Nestlé', 'parent_header': 'Human Resources Policy', 'sub_child_header': '', 'doc_source': 'nestle_hr_policy'}, page_content='## Joining Nestlé  \nThe long-term success of the Company depends on its capacity to attract, retain and develop employees able to ensure its growth on a continuing basis. This is a primary responsibility for all managers.  \nThe Nestlé policy is to hire staff with personal attitudes and professional skills enabling them to develop a long-term relationship with the Company.  \nTherefore the potential for professional development is an essential standard for recruitment.  \nEach new member joining Nestlé is to become a participant in developing a sustainable quality culture which implies a commitment to the organisation, a sense for continuous improvement and leaves no place for complacency.  \nTherefore, and in view of the importance of these Nestlé values, special attention will be paid to the matching between a candid

In [118]:

header_docs = []
seen_headers = set()

for chunk in chunks:
    parent = chunk.metadata.get("parent_header", "").strip()
    child = chunk.metadata.get("child_header", "").strip()

    if not child:
        continue

    combo_key = (doc_id, parent, child)
    if combo_key in seen_headers:
        continue
    seen_headers.add(combo_key)

    searchable_text = f"Policy Section: {child}. Document: {parent}. Rules, guidelines, and policies regarding {child}."

    header_docs.append(
        Document(
            page_content=searchable_text,
            metadata={
                "doc_source": doc_id,
                "parent_header": parent,
                "child_header": child,
                "sub_child_header": "",  
            },
        )
    )


In [119]:
header_docs[0]

Document(metadata={'doc_source': 'nestle_hr_policy', 'parent_header': 'Human Resources Policy', 'child_header': 'Table of Contents', 'sub_child_header': ''}, page_content='Policy Section: Table of Contents. Document: Human Resources Policy. Rules, guidelines, and policies regarding Table of Contents.')

In [ ]:

chunks_vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=index_name,
    namespace="chunks",
)

headers_vectorstore = PineconeVectorStore.from_documents(
    documents=header_docs,
    embedding=embeddings,
    index_name=index_name,
    namespace="headers",
)


In [7]:
from langchain_pinecone import PineconeVectorStore

index_name = "employee-rag-pipeline"

headers_vectorstore = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings,
    namespace="headers",
)

chunks_vectorstore = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings,
    namespace="chunks",
)

test_query = "can an employee wear smart casual"

results2 = headers_vectorstore.similarity_search_with_score(test_query, k=1)

results2

[(Document(id='56e0319f-350f-4f03-b240-040cf3d10267', metadata={'child_header': 'Employee Code of Conduct', 'parent_header': 'Employee Handbook', 'sub_child_header': 'Dress code'}, page_content='Policy Topic: Dress code. Category: Employee Code of Conduct. Document: Employee Handbook. Details, rules, and guidelines regarding Dress code.'),
  0.475557238)]

In [8]:
test_query3 = "what is the work hourse for me"

results3 = chunks_vectorstore.similarity_search_with_score(test_query3, k=1)
results3

[(Document(id='da72bd13-0a9c-453c-95f9-8ca7ffbb77be', metadata={'child_header': 'Time', 'parent_header': 'Employee Handbook', 'sub_child_header': 'Working hours'}, page_content='### Working hours  \nOur company operates between [ _9 a.m. to 7 p.m. on weekdays_ .] You may come to work at any time between [ _9 a.m. and 11 a.m._ ], depending on your team’s needs.  \nSome departments may work after hours, too [ _(e.g. customer support, shipping.)_ ] If you work in these departments, you will follow a shift schedule as needed.'),
  0.498832822)]

In [9]:
test_query4 = "what are all harassment policies"

results4 = chunks_vectorstore.similarity_search_with_score(test_query4, k=1)
results4

[(Document(id='54d64d8b-01dd-4db1-8763-dbc5f50701c3', metadata={'child_header': 'Workplace policies', 'parent_header': 'Employee Handbook', 'sub_child_header': ''}, page_content='## Workplace policies  \nThis section describes policies that apply to everyone at our company: employees, contractors, volunteers, vendors and stakeholders alike. These policies help us build a productive, lawful and pleasant workplace.'),
  0.560312629)]

In [9]:
print(chunks[2])

page_content='## Getting to know our company  
[ _Provide a brief history of your company here. If possible, use illustrations and charts to make this information memorable. Present your company’s culture and vision. Avoid jargon and overused phrases (e.g. “competitive,” “fast-paced environment,” “core values” and “innovative.”) Also, it’d be a good idea to include quotes and pictures of your company’s founders, CEO or other executives to make this introduction more personable._ ]' metadata={'parent_header': 'Employee Handbook', 'child_header': 'Getting to know our company', 'sub_child_header': ''}


In [10]:
print(chunks2[2])

page_content='## A shared Responsibility  
Each employee has a distinct responsibility in dealing with people be it as a leader of a team or as a peer.  
HR managers and their staff are there to provide professional support in handling people matters but should not substitute themselves to the responsible manager. Their prime responsibility is to contribute actively to the quality of HR management throughout the organisation by proposing adequate policies, ensuring their consistent application and coherent implementation with fairness.  
Furthermore, she/he proposes best practices and provides state-of-the-art support and counselling to her/his colleagues. Together they act as co-responsible partners for all HR matters.  
This partnership is the key for efficiency in people management.  
The communication skills of the HR staff must be appropriate to deal with all delicate matters as they occur frequently in human relations issues. They gain their credibility not only from their profes

In [16]:
chunks_combined = chunks + chunks2


In [27]:
print(len(chunks_combined))

59


In [28]:
chunks_combined[58]

Document(metadata={'child_header': 'HR Organisation', 'parent_header': 'Human Resources Policy', 'sub_child_header': '', 'doc_source': 'nestle_hr_policy'}, page_content='## HR Organisation  \nThe fact that Nestlé is more people and product than systems oriented is reflected in the way HR is functioning and is organised.  \nProcesses and systems as well as professional HR tools are there to support HR management but never to the detriment of the human dimension. The human perspective should be present at all times and under all circumstances.  \nThe HR function should report to the manager responsible for a defined operation (Region, Market, Country, Factory) with a functional relation to the market HR according to the size of that operation.  \nWhereas the HR function should indeed provide flawless administrative support, its main role is to add value to the business and to play a proactive role in every situation where HR action is required.  \nSpecific HR KPI’s are useful in assessin

In [32]:
print(type(chunks[0]))

<class 'langchain_core.documents.base.Document'>


In [ ]:
chunk_texts = [doc.page_content for doc in chunks]


In [34]:
chunk_texts[0]

'# Employee Handbook  \n|**Welcome**|**4**|\n|---|---|\n|**Getting to know our company**|**4**|\n|**Employment basics**|**5**|\n|Employment contract types|5|\n|Equal opportunity employment|5|\n|Recruitment and selection process|6|\n|Background checks|6|\n|Referrals|7|\n|Attendance|8|\n|**Workplace policies**|**8**|\n|Confidentiality and data protection|8|\n|Harassment and violence|9|\n|Workplace harassment|10|\n|Workplace violence|10|\n|Workplace safety and health|11|\n|Preventative action|12|\n|Emergency management|12|\n|Smoking|12|\n|Drug-free workplace|13|\n|**Employee Code of Conduct**|**14**|\n|Dress code|14|\n|Cyber security and digital devices|14|\n|Internet usage|15|\n|Cell phone|15|\n|Corporate email|16|\n|Social media|16|\n|Conflict of interest|17|\n|Employee relationships|18|\n|Fraternization|18|\n|Employment of relatives|19|\n|Workplace visitors|19|\n|Solicitation and distribution|20|\n|**Compensation & development**|**20**|\n|Compensation status|20|\n|Overtime|21|\n|Payrol

In [35]:
from pinecone_text.sparse import BM25Encoder
bm25_encoder = BM25Encoder.default()
bm25_encoder.fit(chunk_texts)


100%|██████████| 47/47 [00:00<00:00, 153.86it/s]


In [36]:
bm25_encoder.dump("bm25_params.json")